<a href="https://colab.research.google.com/github/SruthiProject101-Hub/GuviDSProject/blob/main/Number_Plate_%26_Vehicle_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Install Dependencies

In [1]:
# Install core libraries
!pip install -U ultralytics albumentations opencv-python-headless pillow easyocr pytesseract streamlit pyngrok pandas matplotlib seaborn tqdm
# Install optional Levenshtein for OCR evaluation
!pip install python-Levenshtein
# For Tesseract binary (optional fallback)
!apt-get update && apt-get install -y tesseract-ocr libtesseract-dev

  Using cached python_levenshtein-0.27.1-py3-none-any.whl.metadata (3.7 kB)
  Using cached levenshtein-0.27.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.3 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [346 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InR

In [2]:
# --- STEP 1: Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
!mkdir -p /content/dataset/labels

In [4]:
!unzip "/content/drive/MyDrive/Dataset/DATASET.zip" -d /content/dataset/labels

Archive:  /content/drive/MyDrive/Dataset/DATASET.zip
  inflating: /content/dataset/labels/DATASET/notes.json  
  inflating: /content/dataset/labels/DATASET/labels.cache  
  inflating: /content/dataset/labels/DATASET/classes.txt  
  inflating: /content/dataset/labels/DATASET/labels/073e06b8-00f7e34cb220ddbc_jpg.rf.3cff86854ffa4f2ca24b02bc9684fa4c.txt  
  inflating: /content/dataset/labels/DATASET/labels/09a79222-12-D_NQ_NP_690407-MLM31998594359_082019-W_jpg.rf.db2a1ae53abc6440e9eb2_KnRKvaQ.txt  
  inflating: /content/dataset/labels/DATASET/labels/a97c0760-3-D_NQ_NP_858868-MLM31711653804_082019-W_jpg.rf.ae6f2e7f6d4c6a3364fd78_F9CVQwl.txt  
  inflating: /content/dataset/labels/DATASET/labels/acee66f2-002cff1919a39fe7_jpg.rf.827b320a93b742cea302951f41e1ea13.txt  
  inflating: /content/dataset/labels/DATASET/labels/6a2098ce-010a0c44af2d077e_jpg.rf.70b42e126c7af2ad87e84ce4abaccd9c.txt  
  inflating: /content/dataset/labels/DATASET/labels/38db292e-02c79f997837efd4_jpg.rf.5f2fd2948347303996c2f

In [5]:
!ls -R /content/dataset | head -50


/content/dataset:
labels

/content/dataset/labels:
DATASET

/content/dataset/labels/DATASET:
classes.txt
images
labels
labels.cache
notes.json

/content/dataset/labels/DATASET/images:
0008c91f-004ce6e46f66a306_jpg.rf.2f31b4645bfe50921c188146e4066293.jpg
06f88a9e-00b4d07f7be89f98_jpg.rf.261e5fb7b4be3bd9a0cfe6f2f1d2a45d.jpg
073e06b8-00f7e34cb220ddbc_jpg.rf.3cff86854ffa4f2ca24b02bc9684fa4c.jpg
07edd43c-004be6e7ea1c7ff8_jpg.rf.67903af5a80cc47674753b1d52a36091.jpg
08def6e7-7-D_NQ_NP_619911-MLM31423405963_072019-W_jpg.rf.4c114f864305cb3754689b_bAxYwyt.jpg
09a79222-12-D_NQ_NP_690407-MLM31998594359_082019-W_jpg.rf.db2a1ae53abc6440e9eb2_KnRKvaQ.jpg
0e5c7864-2-s-l640_jpg.rf.6e59b4803e3f2982bed6b0fe5a290fd8.jpg
1242f60b-10-cdmx2017taxi_jpg.rf.1ab893cd51a58266b66369d477e4070f.jpg
12e6c43a-01e846c846a1509e_jpg.rf.fe4ac1b73438776a4b7d6ffacafe0a81.jpg
13921c34-01f32030f855c316_jpg.rf.76b2e284650f9ddd3a95a49dd5da5ec6.jpg
1435aeb5-02b3be99914d752c_jpg.rf.f05bd971b32748474ae99de03266451a.jpg
17cec20b-00

In [6]:
#Splitting data for the correct YOLO Structure

import os

base = "/content/dataset/labels/DATASET"
split_base = "/content/dataset/yolo_dataset"

# Make split directories
for split in ["train", "val", "test"]:
    os.makedirs(f"{split_base}/images/{split}", exist_ok=True)
    os.makedirs(f"{split_base}/labels/{split}", exist_ok=True)

print("YOLO directories ready ✅")


YOLO directories ready ✅


In [7]:
import glob, shutil, random

images = glob.glob(f"{base}/images/*.jpg")
random.shuffle(images)

n = len(images)
train_end = int(0.8 * n)
val_end = int(0.9 * n)

splits = {
    "train": images[:train_end],
    "val": images[train_end:val_end],
    "test": images[val_end:]
}

for split, img_list in splits.items():
    for img_path in img_list:
        fname = os.path.basename(img_path)
        label_path = os.path.join(base, "labels", fname.replace(".jpg", ".txt"))

        if os.path.exists(label_path):
            shutil.copy(img_path, f"{split_base}/images/{split}/{fname}")
            shutil.copy(label_path, f"{split_base}/labels/{split}/{os.path.basename(label_path)}")

print("Dataset split complete ✅")


Dataset split complete ✅


In [8]:
#Creating data.yaml

data_yaml = f"""
train: {split_base}/images/train
val:   {split_base}/images/val
test:  {split_base}/images/test

nc: 6
names: ['Number Plate','Car','Truck','Bus','Bike','Person']
"""

with open(f"{split_base}/data.yaml","w") as f:
    f.write(data_yaml)

print(open(f"{split_base}/data.yaml").read())



train: /content/dataset/yolo_dataset/images/train
val:   /content/dataset/yolo_dataset/images/val
test:  /content/dataset/yolo_dataset/images/test

nc: 6
names: ['Number Plate','Car','Truck','Bus','Bike','Person']



## DATA PREPROCESSING

#YOLOv8 automatically resizes input images to the imgsz and normalizes pixel values internally. So manual resizing is optional, but sometimes to standardize resolution to save GPU memory, we can manually resize

In [9]:
#Manually resizing

from PIL import Image
import glob, os

img_folder = "/content/dataset/yolo_dataset/images/train"
target_size = (640, 640)  # height, width

for img_path in glob.glob(os.path.join(img_folder, "*.jpg")):
    img = Image.open(img_path).convert("RGB")
    img = img.resize(target_size)
    img.save(img_path)


In [10]:
#Filtering Poor Quality images

import cv2

def is_blurry(img_path, threshold=100):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    fm = cv2.Laplacian(img, cv2.CV_64F).var()
    return fm < threshold  # True means image is blurry

img_folder = "/content/dataset/yolo_dataset/images/train"
for img_path in glob.glob(os.path.join(img_folder, "*.jpg")):
    if is_blurry(img_path):
        print(f"Removing blurry image: {img_path}")
        os.remove(img_path)
        label_path = img_path.replace("images", "labels").replace(".jpg", ".txt")
        if os.path.exists(label_path):
            os.remove(label_path)


Removing blurry image: /content/dataset/yolo_dataset/images/train/193d31ad-4-D_Q_NP_771917-MLM26259169217_102017-Q_jpg.rf.a0c5bb1ee0207323246b1d9_CvSpb3i.jpg
Removing blurry image: /content/dataset/yolo_dataset/images/train/3e66598f-11-D_Q_NP_3941-MLM4882174000_082013-Q_jpg.rf.d38c91456bef823da5e9ab84342e3c3d.jpg


## MODEL TRAINING


In [11]:
#Install YOLOv8

!pip install -U ultralytics


In [12]:
#Model Training

from ultralytics import YOLO

# Load YOLOv8n model (pretrained)
model = YOLO("yolov8n.pt")

# Train the model
model.train(
    data="/content/dataset/yolo_dataset/data.yaml",  # path to your data.yaml
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    name="vehicle_plate_run",
    augment=True
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.199 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d47a183c410>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

## Post-processing and OCR extraction

In [13]:
#Installing EasyOCR and dependencies

!pip install easyocr
!pip install opencv-python-headless


In [16]:
from ultralytics import YOLO

# Load the trained YOLOv8 model
model = YOLO('/content/runs/detect/vehicle_plate_run/weights/best.pt')

In [17]:
import cv2
import easyocr
import os
import pandas as pd
from ultralytics import YOLO

# Paths
test_folder = "/content/dataset/yolo_dataset/images/test"
plates_folder = "/content/plates"
os.makedirs(plates_folder, exist_ok=True)

# Load YOLOv8 model
model = YOLO('/content/runs/detect/vehicle_plate_run/weights/best.pt')

# Initialize OCR
reader = easyocr.Reader(['en'])

# Prepare a list to store results
results_list = []

# Loop over all test images
for img_file in os.listdir(test_folder):
    img_path = os.path.join(test_folder, img_file)
    results = model.predict(source=img_path, conf=0.1, save=False) # Lowering confidence threshold

    for i, box in enumerate(results[0].boxes.xyxy):
        cls = int(results[0].boxes.cls[i])
        name = model.names[cls]

        if name == "Number Plate":
            x1, y1, x2, y2 = map(int, box)
            img = cv2.imread(img_path)
            plate_crop = img[y1:y2, x1:x2]

            # --- IMAGE PREPROCESSING ---
            gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            enhanced = clahe.apply(gray)
            _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

            # --- OCR ---
            text = reader.readtext(binary)
            plate_text = " ".join([t[1] for t in text])

            # Save cropped plate
            plate_filename = f"{os.path.splitext(img_file)[0]}_plate_{i}.jpg"
            plate_path = os.path.join(plates_folder, plate_filename)
            cv2.imwrite(plate_path, plate_crop)

            # Log result
            results_list.append({
                "image_file": img_file,
                "plate_file": plate_filename,
                "ocr_text": plate_text
            })

# Save results to CSV
results_df = pd.DataFrame(results_list)
results_df.to_csv("/content/plates/ocr_results.csv", index=False)
print("OCR extraction complete. Results saved to /content/plates/ocr_results.csv")
results_df.head()

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete
image 1/1 /content/dataset/yolo_dataset/images/test/c5b6bffd-1_jpg.rf.eeea954cac78b8a206786a2f8b81c471.jpg: 448x640 1 Truck, 1 Bus, 40.4ms
Speed: 1.9ms preprocess, 40.4ms inference, 7.1ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 /content/dataset/yolo_dataset/images/test/cd786d4a-5-20Ways-20To-20Upgrade-20Your-20Car-20License-20Plate-20-_y_jpg.rf.7e_J6jpPcR.jpg: 384x640 1 Truck, 1 Bus, 42.5ms
Speed: 1.8ms preprocess, 42.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/dataset/yolo_dataset/images/test/ed776929-00fb62be50bc2353_jpg.rf.3003dbc5011b37d769d8aa81d72658e0.jpg: 512x640 1 Truck, 42.3ms
Speed: 4.1ms preprocess, 42.3ms inference, 1.5ms postprocess per image at shape (1, 3, 512, 640)

image 1/1 /content/dataset/yolo_dataset/images/test/5766aada-00c96e0e060aeb2f_jpg.rf.be03c2c77534b5825a7d02437d4f82bc.jpg: 352x640 1 Car, 1 Bus, 46.6ms
Speed: 2.7ms 

""


In [18]:
# List the contents of the runs directory to find the correct path to the trained model weights.
!ls -R /content/runs/detect/

/content/runs/detect/:
vehicle_plate_run

/content/runs/detect/vehicle_plate_run:
args.yaml			 labels.jpg	     train_batch202.jpg
BoxF1_curve.png			 results.csv	     train_batch2.jpg
BoxP_curve.png			 results.png	     val_batch0_labels.jpg
BoxPR_curve.png			 train_batch0.jpg    val_batch0_pred.jpg
BoxR_curve.png			 train_batch1.jpg    weights
confusion_matrix_normalized.png  train_batch200.jpg
confusion_matrix.png		 train_batch201.jpg

/content/runs/detect/vehicle_plate_run/weights:
best.pt  last.pt


In [20]:
#Detect, extract number plates, contrast enhancement / binarization

import cv2
import easyocr
import os
from ultralytics import YOLO

# Load model
model = YOLO('/content/runs/detect/vehicle_plate_run/weights/best.pt')
reader = easyocr.Reader(['en'])

# Make folder for cropped plates
os.makedirs("/content/plates", exist_ok=True)

# Pick one test image
test_folder = "/content/dataset/yolo_dataset/images/test"
test_image = os.path.join(test_folder, os.listdir(test_folder)[0])
print("Using test image:", test_image)

# Run detection
results = model.predict(source=test_image, conf=0.5, save=False)

# Process detected plates
for i, box in enumerate(results[0].boxes.xyxy):
    cls = int(results[0].boxes.cls[i])
    name = model.names[cls]

    if name == "Number Plate":
        x1, y1, x2, y2 = map(int, box)
        img = cv2.imread(test_image)
        plate_crop = img[y1:y2, x1:x2]

        # --- IMAGE PREPROCESSING FOR OCR ---
        gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced = clahe.apply(gray)
        _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # --- OCR ---
        text = reader.readtext(binary)
        plate_text = " ".join([t[1] for t in text])
        print(f"Detected Plate {i}: {plate_text}")

        # Save cropped plate
        plate_path = f"/content/plates/plate_{i}.jpg"
        cv2.imwrite(plate_path, plate_crop)


Using test image: /content/dataset/yolo_dataset/images/test/c5b6bffd-1_jpg.rf.eeea954cac78b8a206786a2f8b81c471.jpg

image 1/1 /content/dataset/yolo_dataset/images/test/c5b6bffd-1_jpg.rf.eeea954cac78b8a206786a2f8b81c471.jpg: 448x640 1 Truck, 1 Bus, 8.4ms
Speed: 2.4ms preprocess, 8.4ms inference, 1.7ms postprocess per image at shape (1, 3, 448, 640)


## Lets extend this to loop over all test images, automatically detect all plates, enhance them, and save both cropped plates and OCR results in a CSV . This will be perfect for downstream analysis or dashboard integration.

In [21]:
import cv2
import easyocr
import os
import pandas as pd
from ultralytics import YOLO

# Paths
test_folder = "/content/dataset/yolo_dataset/images/test"
plates_folder = "/content/plates"
os.makedirs(plates_folder, exist_ok=True)

# Load YOLOv8 model
model = YOLO('/content/runs/detect/vehicle_plate_run/weights/best.pt')

# Initialize OCR
reader = easyocr.Reader(['en'])

# Prepare a list to store results
results_list = []

# Loop over all test images
for img_file in os.listdir(test_folder):
    img_path = os.path.join(test_folder, img_file)
    results = model.predict(source=img_path, conf=0.5, save=False)

    for i, box in enumerate(results[0].boxes.xyxy):
        cls = int(results[0].boxes.cls[i])
        name = model.names[cls]

        if name == "Number Plate":
            x1, y1, x2, y2 = map(int, box)
            img = cv2.imread(img_path)
            plate_crop = img[y1:y2, x1:x2]

            # --- IMAGE PREPROCESSING ---
            gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            enhanced = clahe.apply(gray)
            _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

            # --- OCR ---
            text = reader.readtext(binary)
            plate_text = " ".join([t[1] for t in text])

            # Save cropped plate
            plate_filename = f"{os.path.splitext(img_file)[0]}_plate_{i}.jpg"
            plate_path = os.path.join(plates_folder, plate_filename)
            cv2.imwrite(plate_path, plate_crop)

            # Log result
            results_list.append({
                "image_file": img_file,
                "plate_file": plate_filename,
                "ocr_text": plate_text
            })

# Save results to CSV
results_df = pd.DataFrame(results_list)
results_df.to_csv("/content/plates/ocr_results.csv", index=False)
print("OCR extraction complete. Results saved to /content/plates/ocr_results.csv")
results_df.head()



image 1/1 /content/dataset/yolo_dataset/images/test/c5b6bffd-1_jpg.rf.eeea954cac78b8a206786a2f8b81c471.jpg: 448x640 1 Truck, 1 Bus, 6.5ms
Speed: 1.8ms preprocess, 6.5ms inference, 1.3ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 /content/dataset/yolo_dataset/images/test/cd786d4a-5-20Ways-20To-20Upgrade-20Your-20Car-20License-20Plate-20-_y_jpg.rf.7e_J6jpPcR.jpg: 384x640 1 Truck, 1 Bus, 6.4ms
Speed: 1.5ms preprocess, 6.4ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/dataset/yolo_dataset/images/test/ed776929-00fb62be50bc2353_jpg.rf.3003dbc5011b37d769d8aa81d72658e0.jpg: 512x640 1 Truck, 6.6ms
Speed: 2.1ms preprocess, 6.6ms inference, 1.1ms postprocess per image at shape (1, 3, 512, 640)

image 1/1 /content/dataset/yolo_dataset/images/test/5766aada-00c96e0e060aeb2f_jpg.rf.be03c2c77534b5825a7d02437d4f82bc.jpg: 352x640 1 Car, 1 Bus, 6.3ms
Speed: 1.6ms preprocess, 6.3ms inference, 1.1ms postprocess per image at shape (1, 3, 352, 640)

i

""


In [22]:
from google.colab import files
files.download("/content/runs/detect/vehicle_plate_run/weights/best.pt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>